# Object2 MobileNetV1 SSD Training

이 노트북은 `training_code.zip`과 `object2_colab_augmented.zip` 두 압축파일을 업로드한 뒤, 압축 해제부터 Drive 저장 학습과 테스트 이미지 Grad-CAM 확인까지 실행한다.

런타임은 `GPU`로 설정한다. Colab 메뉴에서 `런타임 > 런타임 유형 변경 > T4 GPU`를 선택한다.

In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. 압축파일 업로드
# training_code.zip, object2_colab_augmented.zip 두 파일을 선택한다.
from google.colab import files
uploaded = files.upload()
print('uploaded:', list(uploaded.keys()))

In [ ]:
# 4. 압축 해제 및 폴더 자동 정리
from pathlib import Path
import shutil
import zipfile

WORK = Path('/content/object2_work')
UPLOAD = Path('/content')
TRAIN_DST = WORK / 'training_code'
DATA_DST = WORK / 'object2_colab_augmented'

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True, exist_ok=True)

def unzip_to(zip_name, dst):
    zip_path = UPLOAD / zip_name
    if not zip_path.exists():
        raise FileNotFoundError(f'Missing upload: {zip_path}')
    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dst)

def find_dir(root, required_names):
    required = set(required_names)
    candidates = [root] + [p for p in root.rglob('*') if p.is_dir()]
    for p in candidates:
        names = {child.name for child in p.iterdir()}
        if required.issubset(names):
            return p
    raise FileNotFoundError(f'Cannot find folder containing: {required_names} under {root}')

train_extract = WORK / '_extract_training_code'
data_extract = WORK / '_extract_object2_data'
unzip_to('training_code.zip', train_extract)
unzip_to('object2_colab_augmented.zip', data_extract)

train_src = find_dir(train_extract, ['requirements.txt', 'mbnet', 'pytorch-ssd'])
data_src = find_dir(data_extract, ['Annotations', 'JPEGImages', 'ImageSets', 'labels.txt'])

shutil.copytree(train_src, TRAIN_DST)
shutil.copytree(data_src, DATA_DST)

print('training_code:', TRAIN_DST)
print('dataset:', DATA_DST)
!find /content/object2_work -maxdepth 2 -type d | sort

In [ ]:
# 5. 의존성 설치
%cd /content/object2_work/training_code
!python -m pip install -U pip
!python -m pip install -r requirements.txt

In [ ]:
# 6. 데이터셋 포맷 최종 확인
%cd /content/object2_work/training_code
!python tools/audit_voc_dataset.py ../object2_colab_augmented

In [ ]:
# 7. TensorBoard 먼저 실행
# 학습 로그는 Drive 아래 object2_ssd_runs에 저장된다.
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/object2_ssd_runs

In [ ]:
# 8. 학습 실행
# RUN_NAME은 Drive에 만들어질 실험 폴더 이름이다. 필요하면 바꿔도 된다.
%cd /content/object2_work/training_code
!RUN_NAME=object2_aug_mb1_ssd_v1 bash tools/colab_drive_train_object2.sh

In [ ]:
# 9. 저장 결과 확인
!find /content/drive/MyDrive/object2_ssd_runs -maxdepth 2 -type f | sort | tail -50
!find /content/drive/MyDrive/object2_ssd_runs -maxdepth 3 -type d | sort | tail -50

## 테스트 이미지 추론/Grad-CAM

학습이 끝나면 Drive run 폴더 안의 checkpoint와 같은 폴더의 `labels.txt`를 사용한다. 아래 셀은 `ImageSets/Main/test.txt`의 이미지에 대해 원본, 히트맵, 박스, 원본+히트맵+박스를 한 장으로 저장하고 표시한다.

In [ ]:
# 10. 테스트 이미지 추론 + Grad-CAM 확인
# EPOCH=20은 현재 SGD 학습에서 validation loss가 낮았던 후보다.
# 가장 낮은 validation loss checkpoint를 자동 선택하려면 EPOCH=-1로 바꾼다.
RUN_NAME = 'object2_aug_mb1_ssd_v1'
RUN_DIR = f'/content/drive/MyDrive/object2_ssd_runs/{RUN_NAME}'
EPOCH = 20

%cd /content/object2_work/training_code/mbnet/ros
%run run_colab_gradcam_test_images.py \
  --run-dir "$RUN_DIR" \
  --dataset /content/object2_work/object2_colab_augmented \
  --epoch $EPOCH \
  --split test \
  --num-images 12 \
  --threshold 0.4 \
  --display

In [ ]:
# 11. Grad-CAM 결과 파일 확인
!find "$RUN_DIR/gradcam_test_outputs" -maxdepth 1 -type f | sort | head -30